In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("diabetic_data.csv")
ids = pd.read_csv("IDS_mapping.csv")

print(df.shape)
df.head(3)

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [2]:
# The file has a repeated header pattern — split into 3 sub-tables
# by finding rows where the first column is the column name itself

def extract_mapping(ids_df, id_col_name):
    """Extract a sub-table from the stacked IDS mapping CSV."""
    # Find the header row for this block
    header_idx = ids_df[ids_df.iloc[:, 0].astype(str) == id_col_name].index
    if len(header_idx) == 0:
        return {}
    start = header_idx[0] + 1
    # Find next header row or end of file
    next_headers = ids_df[ids_df.iloc[:, 0].astype(str).str.contains(
        '_id', na=False) & (ids_df.index > start)].index
    end = next_headers[0] if len(next_headers) > 0 else len(ids_df)
    block = ids_df.iloc[start:end].copy()
    block.columns = [id_col_name, 'description']
    block = block.dropna(subset=[id_col_name])
    block[id_col_name] = pd.to_numeric(block[id_col_name], errors='coerce')
    block = block.dropna(subset=[id_col_name])
    block[id_col_name] = block[id_col_name].astype(int)
    return dict(zip(block[id_col_name], block['description']))

admission_type_map      = extract_mapping(ids, 'admission_type_id')
discharge_disp_map      = extract_mapping(ids, 'discharge_disposition_id')
admission_source_map    = extract_mapping(ids, 'admission_source_id')

print("Admission types:", admission_type_map)

Admission types: {}


In [3]:
df.replace('?', np.nan, inplace=True)

# Confirm
print(df.isnull().sum()[df.isnull().sum() > 0])

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


In [4]:
# weight: 97% missing — not usable for dashboarding
# payer_code: 40% missing, not clinically relevant for readmission analysis
# examide / citoglipton: only one unique value (No) — zero variance
cols_to_drop = ['weight', 'payer_code', 'examide', 'citoglipton']
df.drop(columns=cols_to_drop, inplace=True)

print("Remaining columns:", df.shape[1])

Remaining columns: 46


In [5]:
# race: 2.2% missing — fill with 'Unknown'
df['race'] = df['race'].fillna('Unknown')

# medical_specialty: 49% missing — fill with 'Unknown'
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

# diag_1/2/3: small % missing — fill with 'Unknown'
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].fillna('Unknown')

# max_glu_serum / A1Cresult: 83-95% missing — fill with 'Not Tested'
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not Tested')
df['A1Cresult'] = df['A1Cresult'].fillna('Not Tested')

print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


In [6]:
# 'Unknown/Invalid' is not a valid category for dashboarding
df['gender'] = df['gender'].replace('Unknown/Invalid', np.nan)
df['gender'] = df['gender'].fillna('Unknown')

print(df['gender'].value_counts())

gender
Female     54708
Male       47055
Unknown        3
Name: count, dtype: int64


In [7]:
age_midpoint_map = {
    '[0-10)':  5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}

df['age_range'] = df['age']  # keep original for category slicer
df['age_midpoint'] = df['age'].map(age_midpoint_map)

print(df[['age', 'age_range', 'age_midpoint']].drop_duplicates())

        age age_range  age_midpoint
0    [0-10)    [0-10)             5
1   [10-20)   [10-20)            15
2   [20-30)   [20-30)            25
3   [30-40)   [30-40)            35
4   [40-50)   [40-50)            45
5   [50-60)   [50-60)            55
6   [60-70)   [60-70)            65
7   [70-80)   [70-80)            75
8   [80-90)   [80-90)            85
9  [90-100)  [90-100)            95


In [8]:
df['admission_type']         = df['admission_type_id'].map(admission_type_map).fillna('Unknown')
df['discharge_disposition']  = df['discharge_disposition_id'].map(discharge_disp_map).fillna('Unknown')
df['admission_source']       = df['admission_source_id'].map(admission_source_map).fillna('Unknown')

# Keep original IDs for relationships if needed, labels for visuals
print(df[['admission_type_id', 'admission_type']].drop_duplicates().sort_values('admission_type_id'))

       admission_type_id admission_type
1                      1        Unknown
5                      2        Unknown
6                      3        Unknown
2043                   4        Unknown
3089                   5        Unknown
0                      6        Unknown
45829                  7        Unknown
7789                   8        Unknown


In [9]:
# Keep 3-class label for detail
df['readmission_category'] = df['readmitted']  # NO / <30 / >30

# Binary flag for KPI cards and bar charts
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)

# Early readmission flag (<30 days) — clinically significant
df['early_readmission'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

print(df[['readmitted', 'readmission_category', 'readmitted_binary', 'early_readmission']].value_counts())

readmitted  readmission_category  readmitted_binary  early_readmission
NO          NO                    0                  0                    54864
>30         >30                   1                  0                    35545
<30         <30                   1                  1                    11357
Name: count, dtype: int64


In [10]:
# Medication dosage direction: No=0, Steady=1, Up=2, Down=-1
med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': -1}

med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

for col in med_cols:
    if col in df.columns:
        df[col + '_encoded'] = df[col].map(med_map)

# Total number of medications changed (non-zero, non-steady)
encoded_cols = [c + '_encoded' for c in med_cols if c in df.columns]
df['num_med_changes'] = (df[encoded_cols] != 0).sum(axis=1)

print(df[['insulin', 'insulin_encoded', 'num_med_changes']].head())

  insulin  insulin_encoded  num_med_changes
0      No                0                0
1      Up                2                1
2      No                0                1
3      Up                2                1
4  Steady                1                2


In [13]:
def icd9_category(code):
    if pd.isna(code) or code == 'Unknown':
        return 'Unknown'
    code_str = str(code).strip()
    if code_str.startswith('V'):
        return 'Supplementary/V-code'
    if code_str.startswith('E'):
        return 'External Causes'
    try:
        num = float(code_str)
    except:
        return 'Other'
    if num < 140:   return 'Infectious & Parasitic'
    if num < 240:   return 'Neoplasms'
    if num < 280:   return 'Endocrine/Nutritional/Metabolic'
    if num < 290:   return 'Blood Disorders'
    if num < 320:   return 'Mental Disorders'
    if num < 390:   return 'Nervous System'
    if num < 460:   return 'Circulatory'
    if num < 520:   return 'Respiratory'
    if num < 580:   return 'Digestive'
    if num < 630:   return 'Genitourinary'
    if num < 680:   return 'Pregnancy/Childbirth'
    if num < 710:   return 'Skin & Subcutaneous'
    if num < 740:   return 'Musculoskeletal'
    if num < 760:   return 'Congenital'
    if num < 780:   return 'Perinatal'
    if num < 800:   return 'Symptoms/Signs'
    if num < 1000:  return 'Injury & Poisoning'
    return 'Other'

df['diag_1_category'] = df['diag_1'].apply(icd9_category)
df['diag_2_category'] = df['diag_2'].apply(icd9_category)
df['diag_3_category'] = df['diag_3'].apply(icd9_category)

df['primary_diag_diabetes'] = df['diag_1'].apply(
    lambda x: 1 if str(x).startswith('250') else 0
)

print(df['diag_1_category'].value_counts())

diag_1_category
Circulatory                        30336
Endocrine/Nutritional/Metabolic    11459
Respiratory                        10407
Digestive                           9208
Symptoms/Signs                      7636
Injury & Poisoning                  6974
Genitourinary                       5078
Musculoskeletal                     4957
Neoplasms                           3433
Infectious & Parasitic              2768
Skin & Subcutaneous                 2530
Mental Disorders                    2262
Supplementary/V-code                1644
Nervous System                      1211
Blood Disorders                     1103
Pregnancy/Childbirth                 687
Congenital                            51
Unknown                               21
External Causes                        1
Name: count, dtype: int64


In [14]:
df_deduped = df.sort_values('encounter_id').drop_duplicates(
    subset='patient_nbr', keep='first'
).reset_index(drop=True)

print(f"Original:      {len(df)} rows")
print(f"Deduplicated:  {len(df_deduped)} rows")
print(f"Removed:       {len(df) - len(df_deduped)} duplicate patient encounters")

Original:      101766 rows
Deduplicated:  71518 rows
Removed:       30248 duplicate patient encounters
